In [ ]:
import numpy as np
from scipy.integrate import RK45
import matplotlib.pyplot as plt
from collections import deque
import matplotlib.animation as animation

# Constants

In [ ]:
G = 6.6743e-11 # Newton's gravitational constant

# Derivatives

$$
\begin{align*}

&\ddot{x}_{A} = \frac{G m_{B}}{r^{3}} \cdot [x_{B} - x_{A}], &&\ddot{y}_{A} = \frac{G m_{B}}{r^{3}} \cdot [y_{B} - y_{A}] \\\\

&\ddot{x}_{B} = \frac{G m_{A}}{r^{3}} \cdot [x_{A} - x_{B}], &&\ddot{y}_{B} = \frac{G m_{A}}{r^{3}} \cdot [y_{A} - y_{B}] \\\\

&r = \sqrt{(x_{B} - x_{A})^{2} + (y_{B} - y_{A})^2}

\end{align*}
$$

In [ ]:
def calculate_acceleration(m, r, pos_1, pos_2):
    return ((G * m) / r**3) * (pos_2 - pos_1)

def derivatives(t, state, mass_a, mass_b):
    x_a, v_x_a, y_a, v_y_a, x_b, v_x_b, y_b, v_y_b = state

    r = np.sqrt(((x_b - x_a)**2) + ((y_b - y_a)**2))

    acceleration_x_a = calculate_acceleration(mass_b, r, x_a, x_b)
    acceleration_y_a = calculate_acceleration(mass_b, r, y_a, y_b)

    acceleration_x_b = calculate_acceleration(mass_a, r, x_b, x_a)
    acceleration_y_b = calculate_acceleration(mass_a, r, y_b, y_a)

    return [v_x_a, acceleration_x_a, v_y_a, acceleration_y_a, v_x_b, acceleration_x_b, v_y_b, acceleration_y_b]


# Two-Body Orbit Physics

In [ ]:
mass_a = 2e+30 # approx. sun mass
mass_b = 6e+24 # approx. earth mass

orbital_radius = 1.5e11 # approx. 1au (earth-sun distance)
velocity_a = 0.09 # approx. sun orbital velocity
velocity_b = 29800 # approx. earth orbital velocity

orbital_period = (2 * np.pi) * np.sqrt(orbital_radius**3 / (G * mass_a))

dt = orbital_period / 1000 # time step

In [ ]:
solver = RK45(
    fun=lambda t, state: derivatives(t, state, mass_a, mass_b),
    t0=0,
    y0=[
        -orbital_radius * (mass_b/mass_a), # places body a at centre of orbit
        0, 
        0, 
        velocity_a, 
        orbital_radius, 
        0, 
        0, 
        velocity_b
    ],
    t_bound=np.inf,
    max_step=dt
)

# Simulation

In [ ]:
%matplotlib widget

mfig, ax = plt.subplots()

# hide matplotlib widget ui
mfig.canvas.toolbar_visible = False
mfig.canvas.header_visible = False
mfig.canvas.footer_visible = False

# black background colour
ax.set_facecolor('black')
mfig.patch.set_facecolor('black')

# hide axes
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

# set x and y axes to fit orbit
x_limit = orbital_radius * 1.5
y_limit = orbital_radius * 1.5

ax.set_xlim(-x_limit * 1.5, x_limit * 1.5)
ax.set_ylim(-y_limit * 1.5, y_limit * 1.5)
ax.set_aspect("equal")

body_a, = ax.plot([], [], 'o', color='#ff0059', markersize=8)
body_b, = ax.plot([], [], 'o', color='#0091FF', markersize=8)

TRACE_LENGTH = 2000
trace_a_x = deque(maxlen=TRACE_LENGTH)
trace_a_y = deque(maxlen=TRACE_LENGTH)
trace_a, = ax.plot([], [], linewidth=0.5, color="#ff0059")

trace_b_x = deque(maxlen=TRACE_LENGTH)
trace_b_y = deque(maxlen=TRACE_LENGTH)
trace_b, = ax.plot([], [], linewidth=0.5, color="#0091FF")

def update(frame):
    global x_limit, y_limit

    solver.step()
    state = solver.y

    body_a.set_data([state[0]], [state[2]])
    body_b.set_data([state[4]], [state[6]])

    trace_a_x.append(state[0])
    trace_a_y.append(state[2])
    trace_a.set_data(list(trace_a_x), list(trace_a_y))

    trace_b_x.append(state[4])
    trace_b_y.append(state[6])
    trace_b.set_data(list(trace_b_x), list(trace_b_y))

    # adjust x and y limits if body a or b goes out of view
    if(np.abs(state[0]) > x_limit): 
        x_limit = np.abs(state[0])
    
    if(np.abs(state[4]) > x_limit): 
        x_limit = np.abs(state[4])

    if(np.abs(state[2]) > y_limit): 
        y_limit = np.abs(state[2])
    
    if(np.abs(state[6])    > y_limit): 
        y_limit = np.abs(state[6])

    ax.set_xlim(-x_limit * 1.5, x_limit * 1.5)
    ax.set_ylim(-y_limit * 1.5, y_limit * 1.5)


    return body_a, body_b, trace_a, trace_b

anim = animation.FuncAnimation(mfig, update, interval=20, cache_frame_data=False)

plt.show()